In [1]:
import pandas as pd
import re

# -----------------------------
# Load datasets
# -----------------------------
enron = pd.read_csv("Enron.csv")
ceas = pd.read_csv("CEAS_08.csv")


# -----------------------------
# Text cleaning function
# -----------------------------
def clean_text(text):
    text = str(text)

    # Lowercase
    text = text.lower()

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    # Remove leading/trailing whitespace
    text = text.strip()

    return text


# -----------------------------
# Combine subject + body
# -----------------------------
def prepare_dataset(df):
    df = df.copy()

    # Handle missing values
    df["subject"] = df["subject"].fillna("")
    df["body"] = df["body"].fillna("")

    # Combine subject and body
    df["text"] = (
        "subject: "
        + df["subject"].astype(str)
        + " body: "
        + df["body"].astype(str)
    )

    # Clean
    df["text"] = df["text"].apply(clean_text)

    # Remove empty messages
    df = df[df["text"].str.len() > 0]

    # Keep only required columns
    df = df[["text", "label"]]

    # Remove exact duplicate messages
    df = df.drop_duplicates(subset=["text"])

    # Reset index
    df = df.reset_index(drop=True)

    return df


# -----------------------------
# Prepare datasets
# -----------------------------
enron_clean = prepare_dataset(enron)
ceas_clean = prepare_dataset(ceas)


# -----------------------------
# Verify labels
# -----------------------------
print("ENRON")
print(enron_clean.shape)
print(enron_clean["label"].value_counts())
print()

print("CEAS-08")
print(ceas_clean.shape)
print(ceas_clean["label"].value_counts())


# -----------------------------
# Save
# -----------------------------
enron_clean.to_csv("enron_clean.csv", index=False)
ceas_clean.to_csv("ceas08_clean.csv", index=False)

ENRON
(29424, 2)
label
0    15474
1    13950
Name: count, dtype: int64

CEAS-08
(38964, 2)
label
1    21674
0    17290
Name: count, dtype: int64


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load cleaned Enron dataset
enron = pd.read_csv("enron_clean.csv")

# ------------------------------------------------
# First split: 70% train, 30% temporary
# ------------------------------------------------
train_df, temp_df = train_test_split(
    enron,
    test_size=0.30,
    stratify=enron["label"],
    random_state=42
)

# ------------------------------------------------
# Second split: temporary into validation/test
# 50/50 gives us 15% validation and 15% test
# ------------------------------------------------
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

# Reset indices
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# ------------------------------------------------
# Display sizes
# ------------------------------------------------
print("TRAIN")
print(train_df.shape)
print(train_df["label"].value_counts())
print()

print("VALIDATION")
print(val_df.shape)
print(val_df["label"].value_counts())
print()

print("TEST")
print(test_df.shape)
print(test_df["label"].value_counts())

# ------------------------------------------------
# Save
# ------------------------------------------------
train_df.to_csv("enron_train.csv", index=False)
val_df.to_csv("enron_val.csv", index=False)
test_df.to_csv("enron_test.csv", index=False)

TRAIN
(20596, 2)
label
0    10831
1     9765
Name: count, dtype: int64

VALIDATION
(4414, 2)
label
0    2321
1    2093
Name: count, dtype: int64

TEST
(4414, 2)
label
0    2322
1    2092
Name: count, dtype: int64
